# Phase 3 · Nepali (Model L) — Data Generation

Generates a **large** synthetic comparative / transitive reasoning dataset from a
**single unified name pool**, then performs a **random 80 / 10 / 10 split** into
train / val / test. Each example carries an `"answer"` field (the bare entity
name, without trailing punctuation) used for answer-only loss masking during
fine-tuning.

In [1]:
import json
import random
from pathlib import Path

OUTPUT_DIR = Path("../datasets/reasoning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())

Output directory: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/individual-project-nikk8076/nepali/datasets/reasoning


## 1. Unified Entity Pool

A **single** pool of 65 Nepali names (Devanagari script) is used for all splits.
After generation the full dataset is shuffled and split randomly.

In [2]:
# Unified Nepali name pool (65 names)
ALL_NAMES = [
    "राम",     "श्याम",   "सीता",    "गीता",    "हरि",
    "मोहन",    "सावित्री", "कृष्ण",   "सुधा",    "रघु",
    "भानुमती", "विजय",    "पद्मा",   "अर्जुन",  "कमला",
    "नवीन",    "रत्ना",   "प्रकाश",  "शान्ति",  "दुर्गा",
    "श्रीनिवास","माला",   "मुकेश",   "वसन्त",   "दिनेश",
    "कला",     "विक्रम",  "ज्योति",  "सन्तोष",  "सरला",
    "बालाजी",  "रेखा",    "आकाश",    "मीरा",    "रोहित",
    "नलिनी",   "अभि",     "चन्द्र",  "सुरेश",   "निर्मला",
    "गणेश",    "इन्दिरा", "प्रसाद",  "शैलजा",   "रवीन्द्र",
    "उषा",     "अजय",     "सुमित्रा", "तेज",    "मञ्जुला",
    "अमर",     "हेमा",    "देव",     "कान्ता",  "विनोद",
    "स्वाति",  "राहुल",   "प्रभा",   "नवराज",   "विमला",
    "चरण",     "कौशल्या", "दीपा",    "बिमल",    "मनमाया",
]
print(f"Unified name pool: {len(ALL_NAMES)} names")

Unified name pool: 65 names


## 2. Attribute Pools

In [3]:
ATTRIBUTES = {
    "height": {
        "noun": "उचाइ",
        "adj_more": "अग्लो", "adj_less": "होचो",
        "unit": "सेन्टिमिटर",
        "question_more": "को अग्लो छ",
        "question_less": "को होचो छ",
    },
    "age": {
        "noun": "उमेर",
        "adj_more": "जेठो", "adj_less": "कान्छो",
        "unit": "वर्ष",
        "question_more": "को उमेरमा जेठो छ",
        "question_less": "को उमेरमा कान्छो छ",
    },
    "weight": {
        "noun": "तौल",
        "adj_more": "गह्रौं", "adj_less": "हल्का",
        "unit": "किलोग्राम",
        "question_more": "को तौल बढी छ",
        "question_less": "को तौल कम छ",
    },
    "price": {
        "noun": "मूल्य",
        "adj_more": "महँगो", "adj_less": "सस्तो",
        "unit": "रुपैयाँ",
        "question_more": "कुन बढी मूल्यमा बिक्छ",
        "question_less": "कुन कम मूल्यमा बिक्छ",
    },
    "speed": {
        "noun": "गति",
        "adj_more": "छिटो", "adj_less": "ढिलो",
        "unit": "कि.मि./घण्टा",
        "question_more": "को छिटो दौड्यो",
        "question_less": "को ढिलो दौड्यो",
    },
}
print("Attributes:", list(ATTRIBUTES.keys()))

Attributes: ['height', 'age', 'weight', 'price', 'speed']


## 3. Template Functions

Each function returns `(prompt, completion, answer, metadata)` where:
- `completion` = full answer string with trailing period
- `answer` = bare entity name for answer-only loss masking

In [4]:
def t_direct_more(names, attr):
    a, b = names[0], names[1]
    va, vb = random.randint(10, 200), random.randint(10, 200)
    while va == vb: vb = random.randint(10, 200)
    prompt = (f"{a} को {attr['noun']} {va} {attr['unit']} छ. "
              f"{b} को {attr['noun']} {vb} {attr['unit']} छ. "
              f"{attr['question_more']}?")
    ans = a if va > vb else b
    return prompt, f"{ans}.", ans, {"type": "direct_more", "values": {a: va, b: vb}}

def t_direct_less(names, attr):
    a, b = names[0], names[1]
    va, vb = random.randint(10, 200), random.randint(10, 200)
    while va == vb: vb = random.randint(10, 200)
    prompt = (f"{a} को {attr['noun']} {va} {attr['unit']} छ. "
              f"{b} को {attr['noun']} {vb} {attr['unit']} छ. "
              f"{attr['question_less']}?")
    ans = b if va > vb else a
    return prompt, f"{ans}.", ans, {"type": "direct_less", "values": {a: va, b: vb}}

def t_transitive_largest(names, attr):
    a, b, c = names[0], names[1], names[2]
    prompt = (f"{a} को {attr['noun']} {b} भन्दा बढी छ. "
              f"{b} को {attr['noun']} {c} भन्दा बढी छ. "
              f"सबभन्दा {attr['adj_more']} को हो?")
    return prompt, f"{a}.", a, {"type": "transitive_largest", "order": [a, b, c]}

def t_transitive_smallest(names, attr):
    a, b, c = names[0], names[1], names[2]
    prompt = (f"{a} को {attr['noun']} {b} भन्दा बढी छ. "
              f"{b} को {attr['noun']} {c} भन्दा बढी छ. "
              f"सबभन्दा {attr['adj_less']} को हो?")
    return prompt, f"{c}.", c, {"type": "transitive_smallest", "order": [a, b, c]}

def t_transitive_relation(names, attr):
    a, b, c = names[0], names[1], names[2]
    prompt = (f"{a} को {attr['noun']} {b} भन्दा बढी छ. "
              f"{b} को {attr['noun']} {c} भन्दा बढी छ. "
              f"{a} र {c} मध्ये {attr['noun']}मा को अगाडि छ?")
    return prompt, f"{a}.", a, {"type": "transitive_relation", "order": [a, b, c]}

def t_rank_three(names, attr):
    a, b, c = names[0], names[1], names[2]
    va, vb, vc = random.sample(range(10, 200), 3)
    ranked = sorted([(a, va), (b, vb), (c, vc)], key=lambda x: x[1], reverse=True)
    prompt = (f"{a} को {attr['noun']} {va} {attr['unit']}, "
              f"{b} को {attr['noun']} {vb} {attr['unit']}, "
              f"{c} को {attr['noun']} {vc} {attr['unit']} छ. "
              f"सबभन्दा बढी {attr['noun']} भएको को हो?")
    ans_name   = ranked[0][0]
    completion = f"{ans_name}."
    return prompt, completion, ans_name, {"type": "rank_three", "values": {a: va, b: vb, c: vc}}

def t_multihop_largest(names, attr):
    a, b, c, d = names[:4]
    prompt = (f"{a} को {attr['noun']} {b} भन्दा बढी छ. "
              f"{b} को {attr['noun']} {c} भन्दा बढी छ. "
              f"{c} को {attr['noun']} {d} भन्दा बढी छ. "
              f"सबभन्दा बढी {attr['noun']} भएको को हो?")
    return prompt, f"{a}.", a, {"type": "multihop_largest", "order": [a, b, c, d]}

def t_multihop_smallest(names, attr):
    a, b, c, d = names[:4]
    prompt = (f"{a} को {attr['noun']} {b} भन्दा बढी छ. "
              f"{b} को {attr['noun']} {c} भन्दा बढी छ. "
              f"{c} को {attr['noun']} {d} भन्दा बढी छ. "
              f"सबभन्दा कम {attr['noun']} भएको को हो?")
    return prompt, f"{d}.", d, {"type": "multihop_smallest", "order": [a, b, c, d]}

TEMPLATES_2 = [t_direct_more, t_direct_less]
TEMPLATES_3 = [t_transitive_largest, t_transitive_smallest, t_transitive_relation, t_rank_three]
TEMPLATES_4 = [t_multihop_largest, t_multihop_smallest]
print(f"Templates: {len(TEMPLATES_2)} two-entity, {len(TEMPLATES_3)} three-entity, {len(TEMPLATES_4)} four-entity")

Templates: 2 two-entity, 4 three-entity, 2 four-entity


## 4. Dataset Generation + Random Split

Generate **20,000 examples** from the unified pool, shuffle with fixed seed,
then split **80 / 10 / 10** -> 16,000 train / 2,000 val / 2,000 test.

In [5]:
TOTAL   = 20_000
TRAIN_N = 16_000
VAL_N   = 2_000
TEST_N  = 2_000
SEED    = 42

random.seed(SEED)

def generate_all(name_pool, n):
    attr_keys = list(ATTRIBUTES.keys())
    examples  = []
    while len(examples) < n:
        attr_key = random.choice(attr_keys)
        attr     = ATTRIBUTES[attr_key]
        r        = random.random()
        if r < 0.30:
            func   = random.choice(TEMPLATES_2)
            sample = random.sample(name_pool, 2)
        elif r < 0.70:
            func   = random.choice(TEMPLATES_3)
            sample = random.sample(name_pool, 3)
        else:
            func   = random.choice(TEMPLATES_4)
            sample = random.sample(name_pool, 4)
        try:
            prompt, completion, answer, meta = func(sample, attr)
            examples.append({
                "prompt":      prompt,
                "completion":  completion,
                "answer":      answer,
                "language":    "ne",
                "attribute":   attr_key,
                "template_id": func.__name__,
                "metadata":    meta,
            })
        except Exception:
            continue
    return examples

print(f"Generating {TOTAL:,} examples ...")
all_examples = generate_all(ALL_NAMES, TOTAL)
random.shuffle(all_examples)

train_ex = all_examples[:TRAIN_N]
val_ex   = all_examples[TRAIN_N : TRAIN_N + VAL_N]
test_ex  = all_examples[TRAIN_N + VAL_N :]

for split_name, exs in [("train", train_ex), ("val", val_ex), ("test", test_ex)]:
    out_path = OUTPUT_DIR / f"{split_name}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for ex in exs:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"[{split_name:5s}] {len(exs):6,} examples  ->  {out_path}")

Generating 20,000 examples ...
[train] 16,000 examples  ->  ../datasets/reasoning/train.jsonl
[val  ]  2,000 examples  ->  ../datasets/reasoning/val.jsonl
[test ]  2,000 examples  ->  ../datasets/reasoning/test.jsonl


## 5. Dataset Statistics

In [6]:
print("=== Nepali Reasoning Dataset Statistics ===\n")
for split_name in ["train", "val", "test"]:
    path     = OUTPUT_DIR / f"{split_name}.jsonl"
    examples = [json.loads(l) for l in open(path, encoding="utf-8")]
    templates, attrs, seen_names = {}, {}, set()
    for ex in examples:
        templates[ex["template_id"]] = templates.get(ex["template_id"], 0) + 1
        attrs[ex["attribute"]]       = attrs.get(ex["attribute"], 0) + 1
        m = ex["metadata"]
        seen_names.update(m.get("order", []) or list(m.get("values", {}).keys()))
    print(f"{split_name} ({len(examples):,} examples, {len(seen_names)} unique names)")
    print(f"  Templates  : { {k: v for k, v in sorted(templates.items())} }")
    print(f"  Attributes : { {k: v for k, v in sorted(attrs.items())} }")
    print()

sample = json.loads(open(OUTPUT_DIR / "train.jsonl", encoding="utf-8").readline())
assert "answer" in sample, "Missing 'answer' field!"
print(f"'answer' field verified.")
print(f"  prompt    = '{sample['prompt'][:70]}...'")
print(f"  completion= '{sample['completion']}'")
print(f"  answer    = '{sample['answer']}'")

=== Nepali Reasoning Dataset Statistics ===

train (16,000 examples, 65 unique names)
  Templates  : {'t_direct_less': 2405, 't_direct_more': 2412, 't_multihop_largest': 2353, 't_multihop_smallest': 2434, 't_rank_three': 1636, 't_transitive_largest': 1635, 't_transitive_relation': 1524, 't_transitive_smallest': 1601}
  Attributes : {'age': 3251, 'height': 3135, 'price': 3260, 'speed': 3197, 'weight': 3157}

val (2,000 examples, 65 unique names)
  Templates  : {'t_direct_less': 312, 't_direct_more': 311, 't_multihop_largest': 298, 't_multihop_smallest': 293, 't_rank_three': 200, 't_transitive_largest': 194, 't_transitive_relation': 198, 't_transitive_smallest': 194}
  Attributes : {'age': 367, 'height': 403, 'price': 430, 'speed': 417, 'weight': 383}

test (2,000 examples, 65 unique names)
  Templates  : {'t_direct_less': 307, 't_direct_more': 309, 't_multihop_largest': 305, 't_multihop_smallest': 280, 't_rank_three': 205, 't_transitive_largest': 205, 't_transitive_relation': 197, 't_tr

## 6. Sample Examples

In [7]:
print("=== 6 Random Training Examples ===\n")
examples = [json.loads(l) for l in open(OUTPUT_DIR / "train.jsonl", encoding="utf-8")]
for ex in random.sample(examples, 6):
    print(f"[{ex['template_id']}] ({ex['attribute']})")
    print(f"  Prompt    : {ex['prompt']}")
    print(f"  Completion: {ex['completion']}   answer='{ex['answer']}'")
    print()

=== 6 Random Training Examples ===



[t_multihop_largest] (age)
  Prompt    : पद्मा को उमेर रत्ना भन्दा बढी छ. रत्ना को उमेर विनोद भन्दा बढी छ. विनोद को उमेर बालाजी भन्दा बढी छ. सबभन्दा बढी उमेर भएको को हो?
  Completion: पद्मा.   answer='पद्मा'

[t_direct_more] (height)
  Prompt    : गीता को उचाइ 179 सेन्टिमिटर छ. मनमाया को उचाइ 75 सेन्टिमिटर छ. को अग्लो छ?
  Completion: गीता.   answer='गीता'

[t_direct_less] (weight)
  Prompt    : शान्ति को तौल 117 किलोग्राम छ. मञ्जुला को तौल 162 किलोग्राम छ. को तौल कम छ?
  Completion: शान्ति.   answer='शान्ति'

[t_rank_three] (height)
  Prompt    : मीरा को उचाइ 137 सेन्टिमिटर, प्रभा को उचाइ 125 सेन्टिमिटर, अभि को उचाइ 165 सेन्टिमिटर छ. सबभन्दा बढी उचाइ भएको को हो?
  Completion: अभि.   answer='अभि'

[t_rank_three] (age)
  Prompt    : श्याम को उमेर 152 वर्ष, सरला को उमेर 143 वर्ष, स्वाति को उमेर 172 वर्ष छ. सबभन्दा बढी उमेर भएको को हो?
  Completion: स्वाति.   answer='स्वाति'

[t_multihop_largest] (price)
  Prompt    : कान्ता को मूल्य चरण भन्दा बढी छ. चरण को मूल्य सुधा भन्दा बढी छ. सुधा को